In [ ]:
import carla, random, pygame, numpy as np

In [ ]:
# Core elements in carla are: actors, blueprints, worlds, and clients
client = carla.Client('localhost', 2000)

# Set the map to Town06. Silence output
try:
    client.load_world('Town06')
except RuntimeError:
    pass  # Ignore timeout or connection errors

In [ ]:
# Get the world object
world = client.get_world()

# Retrieve the spectator object
spectator = world.get_spectator()

# Print id and type of spectator
# print(spectator) 

# Update the spectator's location and rotation to the center lane of our scenario
# default_location = carla.Location(x=17.092312, y=244.485397, z=0.500000)
# default_rotation = carla.Rotation(pitch=360.000000, yaw=1.780838, roll=0.000000)

# Top-down view of our scenario
default_location = carla.Location(x=115, y=244.485397, z=100.000000)
default_rotation = carla.Rotation(pitch=270.000000, yaw=270, roll=0.000000)

spectator.set_transform(carla.Transform(default_location, default_rotation))

In [ ]:
settings = world.get_settings()
# settings.synchronous_mode = True # Enables synchronous mode. THIS BREAKS THE SIM
settings.fixed_delta_seconds = 0.05 # Set a variable time-step
world.apply_settings(settings)

pygame.init(); pygame.font.init()
W, H = 758, 396
# Add RESIZABLE flag to make window resizable
display = pygame.display.set_mode((W, H), pygame.HWSURFACE | pygame.DOUBLEBUF)


In [ ]:
# Spawn our ego and NPC

ego_blueprint = world.get_blueprint_library().find('vehicle.dodge.charger_2020')
ego_spawn_location = carla.Location(x=21, y=244.485397, z=0.500000)
ego_spawn_rotation = carla.Rotation(pitch=0.000000, yaw=0, roll=0.000000)
ego_spawn_point = carla.Transform(ego_spawn_location, ego_spawn_rotation)
ego_blueprint.set_attribute('role_name', 'hero') # Set the role name of the ego vehicle

npc_blueprint = world.get_blueprint_library().find('vehicle.dodge.charger_police_2020')
npc_spawn_location = carla.Location(x=21, y=(244.485397 - 3.5), z=0.500000)
npc_spawn_rotation = carla.Rotation(pitch=0.000000, yaw=0, roll=0.000000)
npc_spawn_point = carla.Transform(npc_spawn_location, npc_spawn_rotation)

ego_vehicle = world.spawn_actor(ego_blueprint, ego_spawn_point)
npc_vehicle = world.spawn_actor(npc_blueprint, npc_spawn_point)



In [ ]:
cam_bp = world.get_blueprint_library().find("sensor.camera.rgb")
cam_bp.set_attribute("image_size_x", str(W))
cam_bp.set_attribute("image_size_y", str(H))
camera = world.spawn_actor(cam_bp, carla.Transform(carla.Location(x=0.8, z=1.7)), attach_to=ego_vehicle)

In [ ]:

def to_surface(image):
    image.convert(carla.ColorConverter.Raw)
    array = np.frombuffer(image.raw_data, dtype=np.uint8)
    array = array.reshape((image.height, image.width, 4))[:, :, :3][:, :, ::-1]
    return pygame.surfarray.make_surface(array.swapaxes(0, 1))

cam_bp = world.get_blueprint_library().find("sensor.camera.rgb")
cam_bp.set_attribute("image_size_x", str(W))
cam_bp.set_attribute("image_size_y", str(H))
camera = world.spawn_actor(cam_bp, carla.Transform(carla.Location(x=0.8, z=1.7)), attach_to=ego_vehicle)

latest_surface = None
def on_image(img):
    global latest_surface
    latest_surface = to_surface(img)

camera.listen(on_image)

clock = pygame.time.Clock()

npc_vehicle.set_autopilot(True)

try:
    while True:
        world.tick()  # since sync on
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                raise KeyboardInterrupt
        if latest_surface is not None:
            display.blit(latest_surface, (0, 0))
        pygame.display.flip()
        clock.tick_busy_loop(60)
finally:
    camera.stop(); camera.destroy(); ego_vehicle.destroy()
    world.apply_settings(settings._replace(synchronous_mode=False))
    pygame.quit()

In [ ]:
# Start camera with PyGame callback
camera.listen(lambda image: image.save_to_disk('out/%06d.png' % image.frame))

In [ ]:
camera.stop()
# camera.destroy()


In [ ]:
# Delete all vehicles

# Get the list of actors
actors = world.get_actors()

# Delete all vehicles
for actor in actors:
    if actor.type_id.startswith('vehicle') or actor.type_id.startswith('sensor'):
        # print(actor.type_id)
        actor.destroy()


In [ ]:

def to_surface(image):
    image.convert(carla.ColorConverter.Raw)
    array = np.frombuffer(image.raw_data, dtype=np.uint8)
    array = array.reshape((image.height, image.width, 4))[:, :, :3][:, :, ::-1]
    return pygame.surfarray.make_surface(array.swapaxes(0, 1))

vehicle_bp = world.get_blueprint_library().filter("vehicle.*")[0]
vehicle = world.spawn_actor(vehicle_bp, world.get_map().get_spawn_points()[0])

cam_bp = world.get_blueprint_library().find("sensor.camera.rgb")
cam_bp.set_attribute("image_size_x", str(W))
cam_bp.set_attribute("image_size_y", str(H))
camera = world.spawn_actor(cam_bp, carla.Transform(carla.Location(x=0.8, z=1.7)), attach_to=vehicle)

latest_surface = None
def on_image(img):
    global latest_surface
    latest_surface = to_surface(img)

camera.listen(on_image)

clock = pygame.time.Clock()
try:
    while True:
        world.tick()  # since sync on
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                raise KeyboardInterrupt
        if latest_surface is not None:
            # Scale the camera image to fit the current window size
            scaled_surface = pygame.transform.scale(latest_surface, (W, H))
            display.blit(scaled_surface, (0, 0))
        pygame.display.flip()
        clock.tick_busy_loop(60)
finally:
    camera.stop(); camera.destroy(); vehicle.destroy()
    world.apply_settings(settings._replace(synchronous_mode=False))
    pygame.quit()

In [ ]:
# Delete all vehicles

# Get the list of actors
actors = world.get_actors()

# Delete all vehicles
for actor in actors:
    if actor.type_id.startswith('vehicle') or actor.type_id.startswith('sensor'):
        # print(actor.type_id)
        actor.destroy()